## 01. Language Detection & Sentiment Analysis

### 학습 목표
1. `detect_dominant_language`로 텍스트의 언어를 자동 감지한다.
2. `detect_sentiment`로 문서 전체의 감성(긍정/부정/중립/혼합)을 분석한다.
3. `batch_detect_sentiment`로 여러 문서를 한 번에 처리한다.

### API 개요
```python
# 언어 감지
comprehend.detect_dominant_language(Text='...')

# 감성 분석
comprehend.detect_sentiment(Text='...', LanguageCode='ko')

# 배치 감성 분석 (최대 25개)
comprehend.batch_detect_sentiment(TextList=[...], LanguageCode='ko')
```

### 감성 결과 구조
```
{
  'Sentiment': 'POSITIVE' | 'NEGATIVE' | 'NEUTRAL' | 'MIXED',
  'SentimentScore': {
    'Positive': 0.98, 'Negative': 0.01,
    'Neutral': 0.01,  'Mixed': 0.00
  }
}
```

In [ ]:
#환경 초기화
import boto3, json
import matplotlib.pyplot as plt
import pandas as pd

#한글 폰트 설정 (SageMaker Studio)
try:
    import koreanize_matplotlib
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'koreanize-matplotlib'])
    import koreanize_matplotlib

# Amazon Comprehend 클라이언트 생성
# 서울(ap-northeast-2) 리전의 Comprehend 서비스와 연결
comprehend = boto3.client('comprehend', region_name='ap-northeast-2')
print('Comprehend 클라이언트 생성 완료 (서울 ap-northeast-2)')

In [ ]:
#detect_dominant_language
texts = [
    '배송이 빠르고 포장도 꼼꼼했어요. 재구매 의사 있습니다.',
    'The fabric quality is amazing for the price. Highly recommend!',
    'サイズが小さめなので、ワンサイズ大きめがおすすめです。',
    'La couleur est un peu differente de la photo, mais je suis satisfaite.',
]

print('언어 감지 결과:')
print('-' * 55)
for text in texts:
    response = comprehend.detect_dominant_language(
        Text=text 
    )
    top = response['Languages'][0]
    lang_code = top['LanguageCode'] 
    score     = top['Score'] 
    print(f'  [{lang_code}] {score:.3f}  {text[:30]}')

In [ ]:
#detect_sentiment
reviews = [
    '이 제품 정말 최고예요! 배송도 빠르고 품질도 훌륭합니다.',
    '완전 실망입니다. 사진과 너무 다르고 품질이 형편없어요.',
    '그냥 평범한 제품이에요. 나쁘지도 좋지도 않습니다.',
    '좋은 점도 있고 아쉬운 점도 있어요. 가격 대비는 괜찮네요.',
]

results = []
for review in reviews:
    response = comprehend.detect_sentiment(
        Text=review,           # ← review
        LanguageCode='ko'    # ← 'ko'
    )
    sentiment = response['Sentiment']        
    scores    = response['SentimentScore']       
    results.append({'text': review[:25], 'sentiment': sentiment, 'scores': scores})
    print(f'  {sentiment:<10} | {review[:30]}')

print('\n상세 점수 (첫 번째 리뷰):')
for k, v in results[0]['scores'].items():
    print(f'  {k:<12}: {v:.4f}')

#시각화
fig, axes = plt.subplots(1, len(results), figsize=(14, 4))
colors = {'POSITIVE':'#2ecc71','NEGATIVE':'#e74c3c','NEUTRAL':'#95a5a6','MIXED':'#f39c12'}
labels = ['Positive','Negative','Neutral','Mixed']
for ax, r in zip(axes, results):
    vals = [r['scores']['Positive'], r['scores']['Negative'],
            r['scores']['Neutral'],  r['scores']['Mixed']]
    bar_colors = [colors[l.upper()] for l in labels]
    ax.bar(labels, vals, color=bar_colors)
    ax.set_title(f"{r['sentiment']}\n{r['text']}...", fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Score')
plt.suptitle('감성 분석 점수 분포', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
#batch_detect_sentiment
batch_texts = [
    '사이즈가 딱 맞고 색상도 화면이랑 똑같아요.',
    '배송이 일주일 넘게 걸렸어요. 너무 느립니다.',
    '가성비 최고입니다. 친구에게도 추천했어요.',
    '주문한 색상과 다른 색이 왔어요. 교환 요청합니다.',
    '품질 대비 가격이 합리적이에요. 만족합니다.',
]

response = comprehend.batch_detect_sentiment(
    TextList=batch_texts,      
    LanguageCode='ko'   
)

rows = []
for item in response['ResultList']:  
    idx  = item['Index']
    sent = item['Sentiment']
    pos  = item['SentimentScore']['Positive']
    neg  = item['SentimentScore']['Negative']
    rows.append({'텍스트': batch_texts[idx][:20], '감성': sent,
                 'Positive': round(pos,3), 'Negative': round(neg,3)})

df = pd.DataFrame(rows)
print(df.to_string(index=False))